<a href="https://colab.research.google.com/github/carolinampessoa/TechChallengeFase5/blob/main/TechChallengeFase5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Metodologia

Este notebook utiliza um modelo multimodal (LLM) para interpretar diagramas de arquitetura de software.  
Não há treinamento supervisionado; o modelo é utilizado como mecanismo de extração semântica.

Os componentes identificados são então analisados via regras STRIDE pré-definidas.

In [26]:
#Instalação de libs
!pip install openai

In [27]:
#Configurar API Key (uso de LLM da OpenAI)
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Digite sua OpenAI API Key: ")

Digite sua OpenAI API Key: ··········


In [28]:
#Importar imagens para validação
from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print("Imagem carregada:", image_path)


Saving Screenshot_1.png to Screenshot_1 (2).png
Imagem carregada: Screenshot_1 (2).png


In [29]:
#
from openai import OpenAI
import base64
import json

client = OpenAI()

def encode_image(path):
    with open(path, "rb") as img:
        return base64.b64encode(img.read()).decode("utf-8")

base64_image = encode_image(image_path)

prompt = """
Analise o diagrama de arquitetura de software presente na imagem.

Identifique todos os componentes do sistema.
Classifique cada componente em uma das categorias:

- user
- server
- database
- api
- external_system

Responda APENAS em JSON no formato:

{
  "components": [
    {"name": "...", "type": "..."}
  ]
}
"""

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ]
)

output_text = response.output_text
print(output_text)


```json
{
  "components": [
    {"name": "Usuários SEI", "type": "user"},
    {"name": "AWS Shield", "type": "server"},
    {"name": "Amazon CloudFront", "type": "server"},
    {"name": "AWS WAF", "type": "server"},
    {"name": "Application Load Balancer (Availability Zone A)", "type": "server"},
    {"name": "Application Load Balancer (Availability Zone B)", "type": "server"},
    {"name": "Application Load Balancer (Availability Zone C)", "type": "server"},
    {"name": "SEI / SIP (API Server)", "type": "server"},
    {"name": "Amazon Elastic File System (NFS) Multi-AZ", "type": "database"},
    {"name": "Amazon RDS (Primary)", "type": "database"},
    {"name": "Amazon RDS (Secondary)", "type": "database"},
    {"name": "Amazon ElastiCache (memcached) Multi-AZ", "type": "database"},
    {"name": "Solr", "type": "server"},
    {"name": "AWS CloudTrail", "type": "external_system"},
    {"name": "AWS Key Management Service", "type": "external_system"},
    {"name": "AWS Backup", "type"

In [30]:
import re

clean_output_text = output_text.strip()

if clean_output_text.startswith("```"):
    clean_output_text = clean_output_text.replace("```json", "").replace("```", "").strip()

data = json.loads(clean_output_text)
components = data["components"]

In [32]:
stride_map = {
    "server": ["Spoofing", "Tampering", "Denial of Service"],
    "database": ["Tampering", "Information Disclosure"],
    "api": ["Spoofing", "Repudiation"],
    "user": ["Spoofing"],
    "external_system": ["Spoofing", "Tampering"]
}

def analyze_stride(components):
    results = []

    for comp in components:
        threats = stride_map.get(comp["type"])
        if threats is None:
          threats = ["Unknown – No STRIDE mapping defined"]
        results.append({
            "component": comp["name"],
            "type": comp["type"],
            "threats": threats
        })

    return results

stride_results = analyze_stride(components)
stride_results


[{'component': 'Usuários SEI', 'type': 'user', 'threats': ['Spoofing']},
 {'component': 'AWS Shield',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Amazon CloudFront',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'AWS WAF',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Application Load Balancer (Availability Zone A)',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Application Load Balancer (Availability Zone B)',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Application Load Balancer (Availability Zone C)',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'SEI / SIP (API Server)',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Amazon 

In [33]:
def generate_report(results):
    lines = []

    for item in results:
        lines.append(f"Componente: {item['component']}")
        lines.append(f"Tipo: {item['type']}")
        lines.append(f"Ameaças STRIDE: {', '.join(item['threats'])}")
        lines.append("-" * 50)

    return "\n".join(lines)

report = generate_report(stride_results)

print(report)


Componente: Usuários SEI
Tipo: user
Ameaças STRIDE: Spoofing
--------------------------------------------------
Componente: AWS Shield
Tipo: server
Ameaças STRIDE: Spoofing, Tampering, Denial of Service
--------------------------------------------------
Componente: Amazon CloudFront
Tipo: server
Ameaças STRIDE: Spoofing, Tampering, Denial of Service
--------------------------------------------------
Componente: AWS WAF
Tipo: server
Ameaças STRIDE: Spoofing, Tampering, Denial of Service
--------------------------------------------------
Componente: Application Load Balancer (Availability Zone A)
Tipo: server
Ameaças STRIDE: Spoofing, Tampering, Denial of Service
--------------------------------------------------
Componente: Application Load Balancer (Availability Zone B)
Tipo: server
Ameaças STRIDE: Spoofing, Tampering, Denial of Service
--------------------------------------------------
Componente: Application Load Balancer (Availability Zone C)
Tipo: server
Ameaças STRIDE: Spoofing, T

In [34]:
counter_prompt = f"""
Considere as seguintes ameaças STRIDE identificadas:

{json.dumps(stride_results, indent=2)}

Sugira contramedidas de segurança para cada componente.
"""

response2 = client.responses.create(
    model="gpt-4.1",
    input=counter_prompt
)

print(response2.output_text)


Claro! Abaixo estão **contramedidas sugeridas** para cada componente e suas respectivas ameaças STRIDE. As sugestões consideram boas práticas de segurança em ambientes baseados em AWS e aplicações Web/Cloud.

---

### 1. Usuários SEI (User)  
**Ameaças: Spoofing**  
**Contramedidas:**  
- **Autenticação forte (MFA):** Forçar autenticação multifator para todos os usuários.  
- **Senhas fortes e políticas de expiração:** Impor requisitos mínimos de complexidade de senha e troca periódica.  
- **Bloqueio de conta e resposta a tentativas falhas:** Bloquear após X tentativas de login malsucedidas.  
- **Uso de SSO e/ou federated identity:** Integrar com sistemas de autenticação seguros e auditados.

---

### 2. AWS Shield (Server)  
**Ameaças: Spoofing, Tampering, Denial of Service**  
**Contramedidas:**  
- **Proteção contra DDoS (Shield Advanced):** Garantir que Shield Advanced está ativo.  
- **Monitoramento de tráfego:** Utilizar AWS CloudWatch para monitorar eventos suspeitos.  
- **Li